# 使用 Milvus 和 DeepSeek 构建 RAG

DeepSeek 帮助开发者使用高性能语言模型构建和扩展 AI 应用。它提供高效的推理、灵活的 API 以及先进的专家混合 (MoE) 架构，用于强大的推理和检索任务。

在本教程中，我们将展示如何使用 Milvus 和 DeepSeek 构建一个检索增强生成 (RAG) 管道。

## 准备工作

### 依赖与环境

In [1]:
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

---

In [2]:
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

### 准备数据

我们使用 Milvus 文档 2.4.x 中的 FAQ 页面作为我们 RAG 中的私有知识库，这是一个简单 RAG 管道的良好数据源。

下载 zip 文件并将文档解压到 `milvus_docs` 文件夹。

**建议在命令行执行下面命令**

In [3]:
#!wget https://github.com/milvus-io/milvus-docs/releases/download/v2.4.6-preview/milvus_docs_2.4.x_en.zip
#!unzip -q milvus_docs_2.4.x_en.zip -d milvus_docs

我们从 `milvus_docs/en/faq` 文件夹加载所有 markdown 文件。对于每个文档，我们简单地使用 "# " 来分割文件中的内容，这样可以大致分离出 markdown 文件中每个主要部分的内容。

In [6]:
from glob import glob

text_lines = []

for file_path in glob("E:/ai/study/fullstack/deepseek-quickstart/deepseek/milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

In [7]:
len(text_lines)

72

### 准备 LLM 和 Embedding 模型

DeepSeek 支持 OpenAI 风格的 API，您可以使用相同的 API 进行微小调整来调用 LLM。

In [8]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com",  # DeepSeek API 的基地址
)

定义一个 embedding 模型，使用 `milvus_model` 来生成文本嵌入。我们以 `DefaultEmbeddingFunction` 模型为例，这是一个预训练的轻量级嵌入模型。

In [10]:
# from pymilvus import model as milvus_model

# embedding_model = milvus_model.DefaultEmbeddingFunction()

from pymilvus import model as milvus_model

# OpenAI国内代理 https://api.apiyi.com/token 
embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
    model_name='text-embedding-3-large', # Specify the model name
    api_key='sk-02cU1gdtVsd8ny6m19B57d08Fb0347029bAf053863609eB7', # Provide your OpenAI API key
    base_url='https://api.apiyi.com/v1',
    dimensions=512
)

生成一个测试嵌入并打印其维度和前几个元素。

In [11]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

512
[-0.02815247  0.00431061 -0.01856995  0.08197021 -0.03164673 -0.05267334
 -0.04876709  0.12487793 -0.02079773  0.0397644 ]


In [12]:
test_embedding_0 = embedding_model.encode_queries(["That is a test"])[0]
print(test_embedding_0[:10])

[-0.00571823  0.0223999  -0.01893616  0.12805176 -0.01248169 -0.07324219
 -0.00276566  0.08618164 -0.04373169  0.03076172]


## 将数据加载到 Milvus

### 创建 Collection

In [17]:
from pymilvus import MilvusClient, DataType

milvus_client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus"
)

collection_name = "my_rag_collection"

关于 `MilvusClient` 的参数：

*   将 `uri` 设置为本地文件，例如 `./milvus.db`，是最方便的方法，因为它会自动利用 Milvus Lite 将所有数据存储在此文件中。
*   如果您有大规模数据，可以在 Docker 或 Kubernetes 上设置性能更高的 Milvus 服务器。在此设置中，请使用服务器 URI，例如 `http://localhost:19530`，作为您的 `uri`。
*   如果您想使用 Zilliz Cloud（Milvus 的完全托管云服务），请调整 `uri` 和 `token`，它们对应 Zilliz Cloud 中的 Public Endpoint 和 Api key。

检查 collection 是否已存在，如果存在则删除它。

In [18]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

创建一个具有指定参数的新 collection。

如果我们不指定任何字段信息，Milvus 将自动创建一个默认的 `id` 字段作为主键，以及一个 `vector` 字段来存储向量数据。一个保留的 JSON 字段用于存储非 schema 定义的字段及其值。

`metric_type` (距离度量类型):
     作用：定义如何计算向量之间的相似程度。
     例如：`IP` (内积) - 值越大通常越相似；`L2` (欧氏距离) - 值越小越相似；`COSINE` (余弦相似度) - 通常转换为距离，值越小越相似。
     选择依据：根据你的嵌入模型的特性和期望的相似性定义来选择。

 `consistency_level` (一致性级别):
     作用：定义数据写入后，读取操作能多快看到这些新数据。
     例如：
         `Strong` (强一致性): 总是读到最新数据，可能稍慢。
         `Bounded` (有界过期): 可能读到几秒内旧数据，性能较好 (默认)。
         `Session` (会话一致性): 自己写入的自己能立刻读到。
         `Eventually` (最终一致性): 最终会读到新数据，但没时间保证，性能最好。
     选择依据：在数据实时性要求和系统性能之间做权衡。

简单来说：
 `metric_type`：怎么算相似。
 `consistency_level`：新数据多久能被读到。

In [20]:


milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

### 插入数据

遍历文本行，创建嵌入，然后将数据插入 Milvus。

这里有一个新字段 `text`，它是在 collection schema 中未定义的字段。它将自动添加到保留的 JSON 动态字段中，该字段在高级别上可以被视为普通字段。

In [21]:
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|█████████████████████████████████████████████████████████| 72/72 [00:00<00:00, 406994.46it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

## 构建 RAG

### 检索查询数据

我们指定一个关于 Milvus 的常见问题。

In [22]:
question = "How is data stored in milvus?"

在 collection 中搜索该问题，并检索语义上最匹配的前3个结果。

In [23]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)

让我们看一下查询的搜索结果

In [24]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.7758786678314209
    ],
    [
        "How does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float

### 使用 LLM 获取 RAG 响应

将检索到的文档转换为字符串格式。

In [25]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [26]:
context

" Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float32, Float16, and BFloat16 vector types.\n\n- Binary vectors: Store

In [27]:
question

'How is data stored in milvus?'

为语言模型定义系统和用户提示。此提示是使用从 Milvus 检索到的文档组装而成的。

In [28]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""

In [29]:
USER_PROMPT

"\n请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。\n<context>\n Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data typ

使用 DeepSeek 提供的 `deepseek-chat` 模型根据提示生成响应。

In [30]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

Based on the provided context, data in Milvus is stored in two main ways: inserted data and metadata. Inserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental logs, supporting object storage backends like MinIO, AWS S3, Google Cloud Storage, and others. Metadata generated within Milvus are stored in etcd.

<translated>根据提供的上下文，Milvus 中的数据以两种主要方式存储：插入的元数据和元数据。插入的数据，包括向量数据、标量数据和集合特定的模式，作为增量日志存储在持久存储中，支持诸如 MinIO、AWS S3、Google Cloud Storage 等对象存储后端。Milvus 内部生成的元数据存储在 etcd 中。</translated>


In [37]:
from glob import glob

text_lines = []

for file_path in glob("E:/ai/study/fullstack/deepseek-quickstart/deepseek/api/*.md", recursive=True):
    with open(file_path, "r",encoding='utf-8') as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

In [38]:
len(text_lines)

30

In [40]:
collection_name = "my_rag_mfd"
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)


data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|█████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 229615.18it/s]


{'insert_count': 30, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], 'cost': 0}

In [65]:
question1 = "介绍一下物权编?"
question2 = "物权编对日常时候的意义？"

In [58]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question1,question2]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)
retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(retrieved_lines_with_distances[0][0]) 
print(retrieved_lines_with_distances[0][1])
print(retrieved_lines_with_distances[1][0])
print(retrieved_lines_with_distances[1][1])
print(retrieved_lines_with_distances[2][0])
print(retrieved_lines_with_distances[2][1])

（二）物权编

###
0.7259788513183594
第一章 一般规定

**第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。

**第二百零五条** 本编调整因物的归属和利用产生的民事关系。

**第二百零六条** 国家坚持和完善社会主义公有制为主体、多种所有制经济共同发展的基本经济制度。
国家巩固和发展公有制经济，鼓励、支持和引导非公有制经济的发展。
国家实行社会主义市场经济，保障一切市场主体的平等法律地位和发展权利。

**第二百零七条** 国家、集体、私人的物权和其他权利人的物权受法律平等保护，任何组织或者个人不得侵犯。

**第二百零八条** 不动产权利的设立、变更、转让和消灭，应当依照法律规定登记。动产物权的设立和转让，应当依照法律规定交付。

**第二百零九条** 不动产物权的设立、变更、转让和消灭，经依法登记，发生效力；未经登记，不发生效力，但是法律另有规定的除外。
依法属于国家所有的自然资源，所有权可以不登记。

**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。
国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。

**第二百一十一条** 当事人申请登记，应当根据不同登记事项提供材料。
申请登记材料以及登记事项相关信息，可以公开查询。

**第二百一十二条** 登记机构应当履行下列职责：
（一）审查申请人提供的材料；
（二）询问申请人；
（三）如实、及时登记；
（四）法律、行政法规规定的其他职责。
申请登记的不动产存在尚未解决的权属争议的，登记机构应当不予登记，并书面告知申请人。

**第二百一十三条** 登记机构不得有下列行为：
（一）要求对不动产进行评估；
（二）以不动产登记为条件收取其他费用；
（三）超出登记职责范围的其他行为。

**第二百一十四条** 不动产物权的设立、变更、转让和消灭，依照法律规定应当登记的，自记载于不动产登记簿时发生效力。

**第二百一十五条** 不动产登记簿由登记机构管理。
不动产登记簿应当采用纸质形式或者电子形式。
不动产登记簿采用电子形式的，应当备份。

**第二百一十六条** 不动产登记簿是物权归属和内容的根据。
不动产登记簿记载的事项与不动产权属证书记载的事项不一致的，除有证据证明不动产登记

In [59]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question1}
</question>
<translated>
</translated>
"""

In [60]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

抱歉，提供的上下文信息中没有包含关于“物权编”的内容，因此无法回答你的问题。

<translated>
Sorry, the provided context does not contain any information about "Property Law" or "物权编", so I am unable to answer your question.
</translated>


In [61]:
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question2}
</question>
<translated>
</translated>
"""
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据提供的上下文信息，无法找到关于“合同编”的回答，因为上下文内容主要涉及Milvus数据存储、向量类型和处理方式，不包含任何与“合同编”相关的信息。因此，我无法从给定上下文中提供答案。

<translated>
Based on the provided context, no answer can be found regarding "合同编" (Contract Law section), as the context primarily discusses Milvus data storage, vector types, and processing methods, and does not contain any information related to "合同编." Therefore, I cannot provide an answer from the given context.
</translated>


In [62]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [63]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question1}
</question>
<translated>
</translated>
"""
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据提供的《中华人民共和国民法典》物权编的片段，可以总结如下：

物权编主要规定了物的归属和利用所产生的民事关系，旨在明确物的所有权，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序。其核心内容包括：

- **基本制度**：物权编坚持社会主义公有制为主体、多种所有制经济共同发展的基本经济制度，并保障国家、集体、私人的物权及其他权利人的物权受法律平等保护。
- **物权的设立与变动**：不动产物权的设立、变更、转让和消灭需依法登记，登记后方发生效力；动产物权的设立和转让则以交付为准。
- **登记制度**：国家实行不动产统一登记制度，强调登记簿的公示公信作用，明确登记错误后的赔偿责任。
- **用益物权**：规定了用益物权人对他人财产依法享有占有、使用和收益的权利，并需办理登记才能生效。
- **善意取得与遗失物**：界定了善意取得制度，规定了拾得遗失物应返还权利人，以及埋藏物、隐藏物等参照相关规定。
- **征收与征用**：国家为公共利益需要可依法征收或征用不动产，但必须给予公平、合理的补偿。

<translated>根据提供的《中华人民共和国民法典》物权编的片段，可以总结如下：

物权编主要规定了物的归属和利用所产生的民事关系，旨在明确物的所有权，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序。其核心内容包括：

- **基本制度**：物权编坚持社会主义公有制为主体、多种所有制经济共同发展的基本经济制度，并保障国家、集体、私人的物权及其他权利人的物权受法律平等保护。
- **物权的设立与变动**：不动产物权的设立、变更、转让和消灭需依法登记，登记后方发生效力；动产物权的设立和转让则以交付为准。
- **登记制度**：国家实行不动产统一登记制度，强调登记簿的公示公信作用，明确登记错误后的赔偿责任。
- **用益物权**：规定了用益物权人对他人财产依法享有占有、使用和收益的权利，并需办理登记才能生效。
- **善意取得与遗失物**：界定了善意取得制度，规定了拾得遗失物应返还权利人，以及埋藏物、隐藏物等参照相关规定。
- **征收与征用**：国家为公共利益需要可依法征收或征用不动产，但必须给予公平、合理的补偿。</translated>


In [66]:
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question2}
</question>
<translated>
</translated>
"""
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据提供的《中华人民共和国民法典》物权编内容，物权编对日常生活的意义主要体现在以下几个方面：

1. **明确物的归属**（第204条）：帮助人们清楚知道财产（如房屋、车辆、家具等）归谁所有，避免因权属不清产生纠纷。
2. **保护权利人的合法权益**（第204条）：保障个人或组织对其财产的占有、使用、收益和处分等权利，不受他人非法侵犯。
3. **平等保护各类物权**（第207条）：国家、集体、私人的物权受法律平等保护，任何组织或个人不得侵犯，体现了对私有财产权的尊重。
4. **规范不动产登记**（第208-222条）：要求不动产权利的设立、变更、转让和消灭需依法登记，保证物权变动的透明性和法律效力，例如买房后必须登记才能获得完整所有权。
5. **保护善意第三人**（第224、233条）：如船舶、航空器等动产未登记，不得对抗善意第三人；善意取得制度保护不知情的购买者，避免因无权处分导致损失。
6. **处理遗失物和发现物**（第234-238条）：规定拾得遗失物应返还权利人，权利人悬赏寻找需支付报酬，发现埋藏物、漂流物等也有相应处理规则，维护社会公序良俗。
7. **征收征用补偿**（第239-242条）：国家因公共利益征收或征用个人房屋、土地时，必须给予公平合理补偿，保障被征收人的居住条件和财产权利。
8. **用益物权规范**（第353-357条）：允许他人使用自己的不动产（如土地、房屋）并获取收益，但需依法登记且不得损害所有权人权益，促进资源有效利用。

总之，物权编通过明确权属、规范变动、保护权益和平衡各方利益，为日常生活提供了法律框架，保障财产交易安全、维护公平正义。

<translated>
The significance of the Property Rights Section for daily life is mainly reflected in the following aspects:

1. **Clarifying ownership of property** (Article 204): Helps people clearly know who owns property (such as houses, vehicles, furniture, etc.), avoiding disputes due to unc